# 第 52 课｜asyncio 受控并发数据获取

本 Notebook 对应「365 天量化金融」第 52 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为可运行、可验证的量化工程教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/52_youtube-QtFJ1eiu_8o/index.html`
- 环境：`pkm-quant`
- 数据：离线模拟网络延迟和逐标的失败
- 状态：可运行学习稿；不代表生产交易系统

## 本节量化问题

`asyncio` 在等待 I/O 时切换任务，适合多标的数据接口。它不加速 CPU 计算，也不能用无限并发绕过服务端限流。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# asyncio、random 都来自 Python 标准库。
import asyncio

## 1. coroutine、await 与事件循环

`async def` 调用后先得到协程对象；只有 `await` 后才真正执行。`await asyncio.sleep` 模拟网络等待，并把控制权交回事件循环。

In [3]:
# 执行一个协程并检查返回结构；Jupyter 已有事件循环，所以直接 await。
async def fetch_one(symbol, delay=0.01):
    await asyncio.sleep(delay)
    return {"symbol": symbol, "rows": 10, "status": "ok"}

coroutine_object = fetch_one("DEMO_A")
print("调用后的对象类型:", type(coroutine_object).__name__)
single_result = await coroutine_object
print("await 结果:", single_result)
assert single_result["status"] == "ok"

调用后的对象类型: coroutine
await 结果: {'symbol': 'DEMO_A', 'rows': 10, 'status': 'ok'}


## 2. Semaphore 限制同时在途请求

即使同时创建很多任务，`async with semaphore` 只允许固定数量进入接口区。下面额外记录 `max_in_flight`，用证据验证并发上限。

In [4]:
# 并发执行 6 个模拟请求，并验证最多只有 2 个同时在途。
state = {"in_flight": 0, "max_in_flight": 0}
semaphore = asyncio.Semaphore(2)

async def limited_fetch(symbol):
    async with semaphore:
        state["in_flight"] += 1
        state["max_in_flight"] = max(state["max_in_flight"], state["in_flight"])
        try:
            await asyncio.sleep(0.015)
            if symbol == "FAIL_A":
                raise TimeoutError("simulated timeout")
            return {"symbol": symbol, "status": "ok", "rows": 10}
        except TimeoutError as exc:
            return {"symbol": symbol, "status": "failed", "rows": 0, "error": str(exc)}
        finally:
            state["in_flight"] -= 1

symbols = ["A", "B", "C", "FAIL_A", "D", "E"]
results = await asyncio.gather(*(limited_fetch(symbol) for symbol in symbols))
status = pd.DataFrame(results).sort_values("symbol").reset_index(drop=True)
display(status)
print("最大同时在途:", state["max_in_flight"])
assert state["max_in_flight"] == 2
assert status["status"].value_counts().to_dict() == {"ok": 5, "failed": 1}

,symbol,status,rows,error
0,A,ok,10,NaN
1,B,ok,10,NaN
2,C,ok,10,NaN
3,D,ok,10,NaN
4,E,ok,10,NaN
5,FAIL_A,failed,0,simulated timeout


最大同时在途: 2


## 3. 超时属于逐请求状态

`asyncio.wait_for` 把单个请求限制在指定时长；一个任务超时不应让成功任务丢失。最终结果显式按代码排序，不能依赖完成顺序。

In [5]:
# 给不同延迟的任务加统一超时，并保留每项状态。
async def with_timeout(symbol, delay, timeout):
    try:
        result = await asyncio.wait_for(fetch_one(symbol, delay), timeout=timeout)
        return result
    except asyncio.TimeoutError:
        return {"symbol": symbol, "rows": 0, "status": "timeout"}

timed_results = await asyncio.gather(
    with_timeout("FAST", 0.005, 0.02),
    with_timeout("SLOW", 0.04, 0.02),
)
timed_status = pd.DataFrame(timed_results).sort_values("symbol")
display(timed_status)
assert timed_status.set_index("symbol").loc["SLOW", "status"] == "timeout"

,symbol,rows,status
0,FAST,10,ok
1,SLOW,0,timeout


## 方法边界与复用结论


- asyncio 适合 I/O-bound，不会加速占满 CPU 的回测循环。
- 并发量应服从数据源条款、限频和本地连接资源。
- 普通脚本可用 `asyncio.run(main())`；Jupyter 已有事件循环，应在单元格直接 `await`。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。